## Scenario 1: A single data scientist participating in an ML competition

MLflow setup:
* Tracking server: no
* Backend store: local filesystem
* Artifacts store: local filesystem

The experiments can be explored locally by launching the MLflow UI.

In [1]:
import mlflow

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'sqlite:////workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlflow.db'


In [3]:
mlflow.search_experiments()

2026/09/15 19:52:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/15 19:52:22 INFO mlflow.store.db.utils: Updating database tables


[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1789501943958, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789501943958, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Creating an experiment and logging a new run

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/09/15 19:52:45 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/09/15 19:52:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


default artifacts URI: '/workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/1/45ed900ce6cf480b8c7130ed5b44f927/artifacts'


In [5]:
print(mlflow.get_tracking_uri())

sqlite:////workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlflow.db


In [6]:
import os
print(os.environ.get("MLFLOW_TRACKING_URI"))

None


In [9]:
[cell for cell in In if "tracking_uri" in cell]

['print(f"tracking URI: \'{mlflow.get_tracking_uri()}\'")',
 'print(mlflow.get_tracking_uri())',
 '[cell for cell in In if "tracking_uri" in cell]',
 '[cell for cell in In if "tracking_uri" in cell]\nimport os\nprint(os.getcwd())',
 '[cell for cell in In if "tracking_uri" in cell]']

In [10]:
import os
print(os.getcwd())

/workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples


In [7]:
mlflow.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/1', creation_time=1789501965096, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789501965096, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1789501943958, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789501943958, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

## différence entre le fichier mlflow.db créé dans /runnin-mlflow-examples et le sous-dossier mlruns (tous les deux créé dans le scenario 1) : pourquoi mlflow.db n'est pas dans le sous dossier mlruns

Bonne observation — et la réponse est un peu contre-intuitive : **`mlflow.db` et `mlruns/` ne sont pas censés cohabiter.** Ce sont deux *alternatives* pour le même rôle, pas deux pièces d'un même système.

### Les deux backend stores possibles

MLflow doit stocker les métadonnées de tes runs (params, metrics, tags, noms d'expériences). Il a deux façons de le faire :

| Backend store | Où ça vit | Quand |
|---|---|---|
| Filesystem | dossier `mlruns/` (fichiers YAML + texte) | par défaut, si tu ne configures rien |
| Base de données | fichier `mlflow.db` (SQLite) | si tu fais `set_tracking_uri("sqlite:///mlflow.db")` |

Donc `mlflow.db` ne peut pas être *dans* `mlruns/`, parce qu'il **remplace** `mlruns/`. C'est comme demander pourquoi ton classeur papier n'est pas rangé dans ton tableur Excel.

### Pourquoi il est là alors ?

Dans le scénario 1, tu ne configures aucun tracking URI → MLflow utilise le filesystem. La preuve est dans ton propre output :

```
default artifacts URI: '/workspaces/.../running-mlflow-examples/mlruns/1/3a89e4b2a2b...'
```

Tes runs partent bien dans `mlruns/`. Le `mlflow.db` présent dans `running-mlflow-examples/` est donc un **résidu** : il a été créé à un moment où tu as lancé quelque chose avec `sqlite:///mlflow.db` (chemin **relatif**) depuis ce dossier — typiquement un `mlflow ui --backend-store-uri sqlite:///mlflow.db` ou un `set_tracking_uri` lancé alors que ton terminal était positionné là.

C'est exactement le piège du chemin relatif : avec trois slashes (`sqlite:///mlflow.db`), SQLite crée le fichier **dans le répertoire courant**, quel qu'il soit. D'où l'habitude que tu as prise d'utiliser quatre slashes et un chemin absolu (`sqlite:////workspaces/...`) — ça force le fichier à toujours atterrir au même endroit.

### Vérification rapide

Dans ton notebook :

```python
print(mlflow.get_tracking_uri())
```

- Si ça affiche `file:///.../running-mlflow-examples/mlruns` → tu es bien en mode filesystem, le `.db` local ne sert à rien ici.
- Si ça affiche `sqlite:///...` → alors c'est l'inverse, et c'est `mlruns/` qui serait le résidu.

Tu peux supprimer ce `mlflow.db` de `running-mlflow-examples/` sans risque **une fois la vérification faite** — mais attention à ne pas toucher à celui qui est dans `02-experiment-tracking/`, qui contient probablement tes expériences de `duration-prediction.ipynb`.

C'est justement le sujet des scénarios 2 et 3 : passer du filesystem à une vraie base + un serveur de tracking. Tu vas voir la différence très concrètement.

remarque perso : la commande affoche sqlite:////workspaces/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlflow.db. --> claude ne sait pas pourquoi ça n'affiche pas mlruns/, j'abandonne

### Interacting with the model registry

In [8]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [9]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")